## `nz_splines_polybias.ipynb`
-------------

In [ ]:
import numpy as np
import importlib
import json
import matplotlib.pyplot as plt

from pathlib import Path

import src.statistics.spline as spline
import src.statistics.corrfiles as cf
import src.statistics.systematics as sy

importlib.reload(spline)
importlib.reload(sy)

ROOT = cf.get_base_dir()

In [ ]:
scale_cut = [0.3, 3]
version = "v_1p1"
names = sy.NAMES

tag = sy.scale_cut_tag(scale_cut)
DATA_DIR = sy.variant_dir(ROOT, "polybias", scale_cut, version)
SPL_DIR = sy.variant_dir(ROOT, "polybias", scale_cut, version, what="splines")
SPL_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_DIR / f"polybias_metadata_{tag}_{version}.json") as f:
    meta = json.load(f)

FIT_KWARGS = dict(
    n_tune=400,
    n_samples=1600,
    target_accept=0.99,
    prior_concentration=3,
    base_alpha=0.05,
)

print(f"bias mode : {meta['bias_mode']} (full covariance: {meta['use_covariance']})")
print(f"Data      : {DATA_DIR}")
print(f"Splines   : {SPL_DIR}")

In [ ]:
data = np.load(DATA_DIR / f"merged_res_norm_{tag}_{version}.npz")
print(sorted(data.files)[:8], "...")

failed = []
for name in names:
    for tomo in sy.TOMO_BINS:
        savefile = str(SPL_DIR / f"spl_{name}_{tomo}")
        if Path(f"{savefile}.nc").exists():
            print(f"Skipping {savefile}, already exists")
            continue

        z = data[f"{tomo}/{name}_z"]
        npz_arr = data[f"{tomo}/{name}"]
        npz_arr_err = data[f"{tomo}/{name}_err"]

        print(f"\n=== {name}, tomo {tomo} ({len(z)} points) ===")
        try:
            spl = spline.BayesianBSpline(zv=z, n_knots=int(len(z) // 2))
            spl.fit(npz_arr, npz_arr_err, **FIT_KWARGS)
            spl.save_model(savefile)
        except Exception as exc:  # noqa: BLE001 -- keep the remaining fits going
            failed.append((name, tomo, repr(exc)))
            print(f"  FAILED {name}, tomo {tomo}: {exc!r}")

if failed:
    print(f"\n{len(failed)} fit(s) failed; re-run this cell to retry just those:")
    for nm, tomo, exc in failed:
        print(f"  {nm}, tomo {tomo}: {exc}")
else:
    print("\nAll fits complete.")

In [ ]:
# sanity check: npz_cross and npz_bs are carried over unchanged from the fiducial run,
# so their inputs must match the fiducial spline inputs exactly.
FID_SPL = ROOT / "results" / f"splines_{tag}_{version}"
for name in ("npz_cross", "npz_bs"):
    f = FID_SPL / f"spl_{name}_1"
    if not Path(f"{f}.nc").exists():
        print(f"No fiducial spline at {f}, skipping check.")
        continue
    fid = spline.BayesianBSpline.from_saved_model(str(f))
    new = spline.BayesianBSpline.from_saved_model(str(SPL_DIR / f"spl_{name}_1"))
    print(f"{name}: max |dn(z)| = {np.abs(fid.nz - new.nz).max():.3e}, "
          f"max |dsigma| = {np.abs(fid.nz_err - new.nz_err).max():.3e}")

In [ ]:
z_all = np.linspace(0, 3, 600)
fig, axs = plt.subplots(2, 2, figsize=(11, 7))
for tomo, ax in zip(sy.TOMO_BINS, axs.flat):
    for name in names:
        f = SPL_DIR / f"spl_{name}_{tomo}"
        if not Path(f"{f}.nc").exists():
            continue
        spl = spline.BayesianBSpline.from_saved_model(str(f))
        mask = (z_all <= spl.zv.max()) & (z_all >= spl.zv.min())
        samples = sy.normalized_samples(spl, z_all[mask])
        ax.plot(z_all[mask], np.percentile(samples, 50, axis=0), lw=1.8, label=name)
    ax.set_title(f"Bin {tomo}")
    ax.set_xlabel("Redshift")
    ax.set_ylabel("n(z)")
    ax.grid(True, alpha=0.3)
    if tomo == 1:
        ax.legend(fontsize=8)
fig.suptitle("Polynomial-bias variant: spline medians")
fig.tight_layout()